In [12]:
import os, pickle

import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import hist
import vector

from physics.analysis import zz4l, zz2l2v
from datasets import balanced
from nsbi import carl
from datasets.balanced import BalancedDataset

import torch
from torch.utils.data import Dataset
import lightning

In [13]:
torch.set_float32_matmul_precision('medium')

matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,  # Don't override with default matplotlib fonts
    "pgf.preamble": "", 
})

In [14]:
output_dir = 'run/h4l/'
events_numerator_test, events_denominator_test, scaler, carl_model = carl.utils.load_results(output_dir)
features = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy', 'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy', 'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy', 'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']
batch_size = 4096

# output_dir = 'run/h2l2v/'
# carl_model = carl.utils.load_results(output_dir)
# features = ["l1_pt", "l1_eta", "l1_phi", "l1_energy", "l2_pt", "l2_eta", "l2_phi", "l2_energy", "met", "met_phi"]
# batch_size = 4096

In [15]:
ds_test = BalancedDataset(events_numerator_test, events_denominator_test, features=features, scaler=scaler)
dl_test = torch.utils.data.DataLoader(torch.tensor(ds_test.X, dtype=torch.float32), batch_size=batch_size)

In [16]:
trainer = lightning.Trainer(accelerator='gpu', devices=1)

# predictions_train = torch.concatenate(trainer.predict(carl_model, dataloaders=[dl_train]), axis=0).detach().numpy()
# predictions_val_batches = trainer.predict(carl_model, dl_val)
predictions_test_batches = trainer.predict(carl_model, dl_test)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

In [17]:
# predictions_val = torch.cat(predictions_val_batches).detach().numpy()
# targets_val = validation_data.s
# weights_val = validation_data.w * len(validation_data)

predictions_test = torch.cat(predictions_test_batches).detach().numpy()
targets_test = ds_test.s
weights_test = ds_test.w * len(ds_test)

In [18]:
s_nbins = 50
s_min, s_max = 0.0, 1.0
s_step = (s_max - s_min)/s_nbins
s_centers = [s_min+(i+1/2)*s_step for i in range(s_nbins)]

In [21]:
# p = predictions_val
# t = targets_val
# w = weights_val

p = predictions_test
t = targets_test
w = weights_test

pred_binned = [p[(p > s_min+s_step*i) & (p <= s_min+s_step*(i+1))] for i in range(s_nbins)]
targets_binned = [t[(p > s_min+s_step*i) & (p <= s_min+s_step*(i+1))] for i in range(s_nbins)]
weights_binned = [w[(p > s_min+s_step*i) & (p <= s_min+s_step*(i+1))] for i in range(s_nbins)]

sig_per_bin = np.array([np.sum((targets_binned[i]==1.0) * weights_binned[i]) for i in range(s_nbins)])
bkg_per_bin = np.array([np.sum((targets_binned[i]==0.0) * weights_binned[i]) for i in range(s_nbins)])
s_true = sig_per_bin/(sig_per_bin+bkg_per_bin)

sig_err = np.sqrt(np.array([np.sum((targets_binned[i]==1.0) * weights_binned[i] * weights_binned[i]) for i in range(s_nbins)]))
bkg_err = np.sqrt(np.array([np.sum((targets_binned[i]==0.0) * weights_binned[i] * weights_binned[i]) for i in range(s_nbins)]))
s_err = np.sqrt(
    (sig_err * bkg_per_bin / (sig_per_bin + bkg_per_bin)**2)**2 +
    (bkg_err * sig_per_bin / (sig_per_bin + bkg_per_bin)**2)**2
)

/tmp/ipykernel_631444/2775720337.py:15: RuntimeWarning: invalid value encountered in divide
  s_true = sig_per_bin/(sig_per_bin+bkg_per_bin)
/tmp/ipykernel_631444/2775720337.py:20: RuntimeWarning: invalid value encountered in divide
  (sig_err * bkg_per_bin / (sig_per_bin + bkg_per_bin)**2)**2 +
/tmp/ipykernel_631444/2775720337.py:21: RuntimeWarning: invalid value encountered in divide
  (bkg_err * sig_per_bin / (sig_per_bin + bkg_per_bin)**2)**2


In [22]:
fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(5,6), sharex=True)
plt.subplots_adjust(wspace=0, hspace=0)
ax1.set_aspect('equal', adjustable='box')

ax1.errorbar(s_centers, s_centers, color='grey', linestyle='--', label='$\\mathrm{MC}$')
ax1.errorbar(s_centers, s_true, xerr=s_step/2, yerr=s_err, color='royalblue', linestyle='none', linewidth=1.2, label='$\\mathrm{NSBI}$')

ax1.set_ylim(s_min, s_max)
ax1.set_ylabel('$\\mathrm{MC\\ estimate}\\ \\frac{p_{q\\bar{q}}(x)}{ p_{q\\bar{q}}(x) + p_{gg}(x) }$', fontsize=15)

ax1.legend(frameon=False, fontsize=12)

ax2.errorbar(s_centers, np.array(s_centers)-np.array(s_centers), color='grey', linestyle='--')
ax2.errorbar(s_centers, np.array(s_true)-np.array(s_centers), xerr=s_step/2, yerr=s_err, color='royalblue', linestyle='none', linewidth=1.2)

ax2.set_xlim(s_min, s_max)
ax2.set_xlabel('$\\mathrm{NSBI\\ estimate}\\ \\hat{s}(x)$', fontsize=15)
ax2.set_ylabel('$\\mathrm{Residual}$', fontsize=15)
ax2.set_ylim(-0.075, 0.075)
ax2.yaxis.set_ticks([-0.05, 0.05])

ax1.tick_params(labelsize=12)
ax2.tick_params(labelsize=12)

plt.tight_layout()
plt.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

ax2.xaxis.set_tick_params(which='both', labeltop=False, top=True)

plt.savefig('carl_calibration.pdf', bbox_inches='tight')
plt.close()

In [23]:
predictions_denominator_test = carl_model(torch.tensor(scaler.transform(events_denominator_test.kinematics[features].to_numpy()), dtype=torch.float32)).detach().numpy().ravel()
# predictions_denominator_val = carl_model(torch.tensor(scaler.transform(events_denominator_val.kinematics[features].to_numpy()), dtype=torch.float32)).detach().numpy().ravel()

In [ ]:
m4l_numerator = events_numerator_test.calculate(zz4l.FourLeptonSystem()).kinematics['4l_mass']
m4l_denominator = events_denominator_test.calculate(zz4l.FourLeptonSystem()).kinematics['4l_mass']
xobs_numerator = m4l_numerator
xobs_denominator = m4l_denominator
xbins = np.concatenate([np.arange(180,250,10), np.arange(250,500,50), np.arange(500,750,125), np.arange(750,1100,250)])
xmin, xmax = 180, 1000
xwidths = np.diff(xbins)
xcenters = xbins[:-1] + xwidths/2
xlabel = '$m_{ZZ}\\ \\mathrm{[GeV]}$'
y2_min, y2_max = 0.5, 3.5
y3_min, y3_max = 0.85, 1.15

# mtzz_numerator = events_numerator_test.kinematics['zz_mt']
# mtzz_denominator = events_denominator_test.kinematics['zz_mt']
# xobs_numerator = mtzz_numerator
# xobs_denominator = mtzz_denominator
# xbins = np.concatenate([np.arange(250,300,10), np.arange(300,400,25), np.arange(400,500,50), np.arange(500,1100,100)])
# xmin, xmax = 250, 1000
# xwidths = np.diff(xbins)
# xcenters = xbins[:-1] + xwidths/2
# xlabel = '$m_{\\mathrm T}^{ZZ}\\ \\mathrm{[GeV]}$'
# y2_min, y2_max = 0.75, 1.75
# y3_min, y3_max = 0.85, 1.15

In [ ]:
lw = 1.25

h_num_mc = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_num_mc.fill(xobs_numerator, weight=events_numerator_test.probabilities)

h_denom = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_denom.fill(xobs_denominator, weight=events_denominator_test.probabilities)

h_num_carl = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_num_carl.fill(xobs_denominator, weight=events_denominator_test.probabilities * (predictions_denominator_test / (1 - predictions_denominator_test)))

fig, (ax1, ax2, ax3) = plt.subplots(3,1,gridspec_kw={'height_ratios': [2, 1, 1]},figsize=(5,6), layout='constrained', sharex=True)

ax1.stairs(h_num_carl.values()/xwidths, h_num_carl.axes[0].edges, color='cornflowerblue', linewidth=lw, label='$q\\bar{q} \\to ZZ\\ (\\mathrm{NSBI})$')
ax1.stairs(h_num_mc.values()/xwidths, h_num_mc.axes[0].edges, color='blue', linestyle='--', linewidth=lw, label='$q\\bar{q} \\to ZZ\\ (\\mathrm{MC})$')
ax1.stairs(h_denom.values()/xwidths, h_denom.axes[0].edges, color='black', linestyle='--', linewidth=lw, label='$gg(\\to h^{\\ast}) \\to ZZ$')

ax2.stairs(h_num_carl.values()/h_denom.values(), h_num_mc.axes[0].edges, color='cornflowerblue', linewidth=lw)
ax2.stairs(h_num_mc.values()/h_denom.values(), h_num_mc.axes[0].edges, color='blue', linestyle='--', linewidth=lw)
ax2.stairs(h_denom.values()/h_denom.values(), h_denom.axes[0].edges, color='black', linestyle='--', linewidth=lw)

ax3.stairs(h_num_mc.values()/h_num_mc.values(), xbins, color='blue', linestyle='--', linewidth=lw)
ax3.errorbar(xcenters, h_num_carl.values()/h_num_mc.values(), yerr=np.sqrt(h_num_carl.variances())/h_num_carl.values(), xerr=xwidths/2, fmt='none', color='cornflowerblue', linewidth=lw)

ax1.legend(frameon=False, fontsize=12)

ax1.set_yscale('log')
ax2.set_ylim(y2_min, y2_max)
ax3.set_ylim(y3_min, y3_max)

ax1.tick_params(labelsize=12)
ax2.tick_params(labelsize=12)
ax3.tick_params(labelsize=12)

ax1.set_ylabel('$\\mathrm{Density\\ of\\ events\\ [1/GeV]}$', fontsize=15, loc='top')
ax2.set_ylabel('$p_{q\\bar{q}}/p_{gg}$', fontsize=15)
ax3.set_ylabel('$\\mathrm{NSBI}/\\mathrm{MC}$', fontsize=12)

ax3.set_xlabel(xlabel, fontsize=12)
ax3.set_xlim(xmin, xmax)

plt.tight_layout()
plt.subplots_adjust(hspace=0)

plt.savefig('carl_reweight.pdf', bbox_inches='tight')

/tmp/ipykernel_475596/834724248.py:42: UserWarning: The figure layout has changed to tight
  plt.tight_layout()
